# Preprocessing Data:
Take the datasets and preprocess them for model use

# Tasks
1. Handle missing values
2. Convert Categorical fields to numeric (encoding)
3. Normalize numerical features
4. Create train/test split
	- Recent month = test set
	- X months immediately preceding = training set
		- X is not fix and can be a tunable choice 
		- Experiment to find optimal value of X

# Importing data once again

In [51]:
import pandas as pd
import numpy as np

# the root directory of project
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"

# Fetch houses sold in between May 2025 and May 2026
houses_sold_by_month = []

for month_after_june in range(0, 13):
    y = "2025" if month_after_june < 7 else "2026"
    m =  ""
    if (month_after_june + 6 < 10):
        m = f"0{6 + month_after_june}"
    elif (month_after_june + 6 >= 10 and month_after_june + 6 < 13): 
       m = f"{6 + month_after_june}" 
    else:
        m = f"0{month_after_june - 6}"
    print(f"../raw_data/CRMLSSold{y}{m}.csv")
    filename = f"{root}/raw_data/california/CRMLSSold{y}{m}.csv"
    houses_sold_by_month.append(pd.read_csv(filename))

# Concatenate dataframes
# Ignoring indexes because I'm sure there are duplicates somewhere in the data
houses_sold_2026 = pd.concat(houses_sold_by_month, ignore_index=True)

# Setting float display property with Pandas
pd.set_option('display.float_format', lambda x: '{:,.4f}'.format(x))
pd.set_option('display.max_rows', None)

../raw_data/CRMLSSold202506.csv
../raw_data/CRMLSSold202507.csv


C:\Users\donutii\AppData\Local\Temp\ipykernel_15264\2485105764.py:21: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  houses_sold_by_month.append(pd.read_csv(filename))


../raw_data/CRMLSSold202508.csv
../raw_data/CRMLSSold202509.csv
../raw_data/CRMLSSold202510.csv
../raw_data/CRMLSSold202511.csv
../raw_data/CRMLSSold202512.csv
../raw_data/CRMLSSold202601.csv


C:\Users\donutii\AppData\Local\Temp\ipykernel_15264\2485105764.py:21: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  houses_sold_by_month.append(pd.read_csv(filename))


../raw_data/CRMLSSold202602.csv
../raw_data/CRMLSSold202603.csv
../raw_data/CRMLSSold202604.csv
../raw_data/CRMLSSold202605.csv
../raw_data/CRMLSSold202606.csv


Column Audit: 
- 

# Delete Bad Datapoints / Remove unhelpful columns

In [52]:
# Functions
def print_change(change, size1, size2):
    print(f'{change}: {size1 - size2}, {percent(size1, size2)}% decrease')
def percent(size1, size2):
    return ((size1 - size2) / size1) * 100

In [53]:
# See columns
print(houses_sold_2026.columns)
print(f'Total size preprocessing: {len(houses_sold_2026)}')

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', '

In [54]:
# Take out strictly unneeded columns
# Mostly datapoints liable for leakage and other metadata not helpful for analysis (along with information only known after purchase)
# Taking out lotsizeacres since residential usually focuses on sqft
# Taking out BusinessType because we're focusing on residential
# Taking LotSizeArea out because it's functionally equivalent to lotSizesquarefeet
houses_sold_2026 = houses_sold_2026.drop(columns=['BuyerAgentAOR', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'ListAgentFirstName','ListAgentLastName', 'ListPrice', 'DaysOnMarket', 
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName','BuyerAgentMlsId', 'BuyerAgentFirstName', 
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MlsStatus', 'LotSizeAcres',  'StreetNumberNumeric', 'ListingId', 
       'ContractStatusChangeDate', 'CoBuyerAgentFirstName', 'PurchaseContractDate', 'ListingContractDate', 'BusinessType', 'LotSizeArea'])

# Not sure I want to track the effect of the listers so I'll drop them :P
houses_sold_2026 = houses_sold_2026.drop(columns=['ListAgentAOR', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'MLSAreaMajor',
                                                  'BuyerOfficeAOR'])

# See columns
print(houses_sold_2026.columns)
print(f'Total size after cutting columns: {len(houses_sold_2026)}')

Index(['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
       'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'FireplacesTotal',
       'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount',
       'CountyOrParish', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'SubdivisionName',
       'YearBuilt', 'BathroomsTotalInteger', 'City', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict',
       'BelowGradeFinishedArea', 'StateOrProvince', 'CoveredSpaces',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool',
       'Levels', 'LotSizeDimensions', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict'],
      dtype='object')
Total size after cutting columns: 283176


In [55]:
# Let model learn only based on the single-family residencies to cut out extreme outliers (Tyler Beede)
# (It is allowed, though I'd rather my data be clean than not)
# ... it is almost a 50% decrease, though. Maybe not. Instead I think I will be more intentional about which types and subtypes to include
# I think I want to include residential lease also just because it feels in the spirit of the research
prev_size = len(houses_sold_2026)
houses_sold_2026 = houses_sold_2026[houses_sold_2026['PropertyType'].isin(['Residential', 'ResidentialLease'])]
#print('Close Price mean of Residential: ')
#print(houses_sold_2026[houses_sold_2026['PropertyType'] == 'Residential']['ClosePrice'].describe())
#print('Close Price mean of ResidentialLease: ')
#print(houses_sold_2026[houses_sold_2026['PropertyType'] == 'ResidentialLease']['ClosePrice'].describe())

# I'm including single family residences, Cabins, Condominiums, Lofts, Dockominiums, and Townhouses
# Nevermind, both only include single family residences
houses_sold_2026 = houses_sold_2026[houses_sold_2026['PropertySubType'].isin(['SingleFamilyResidence'])]
print(houses_sold_2026[['PropertyType', 'PropertySubType']].value_counts())
print_change('After filtering by Single Family Residential Properties', prev_size, len(houses_sold_2026))
# Now there's about a 39% decrease. Not ideal, but I don't mind it as much.
# Checking out outliers
print(houses_sold_2026['ClosePrice'].describe())

PropertyType      PropertySubType      
Residential       SingleFamilyResidence    143078
ResidentialLease  SingleFamilyResidence     30513
Name: count, dtype: int64
After filtering by Single Family Residential Properties: 109585, 38.69854789953951% decrease
count       173,591.0000
mean      1,108,558.2648
std       7,185,154.6432
min               0.0000
25%         430,000.0000
50%         776,000.0000
75%       1,280,000.0000
max     989,500,000.0000
Name: ClosePrice, dtype: float64


In [56]:
# View each column by number of null
null_col_values = houses_sold_2026.isnull().sum()
for col, null_count in null_col_values.items(): 
    total_rows = houses_sold_2026.shape[0]
    pct_null = (null_count / total_rows) * 100
    if pct_null > 90: 
        print(f"Column {col}: {null_count} null values, {pct_null:.2f}% null") 
    # print(f'Column {col}: {null_col_values[col]} null values, {percent(houses_sold_2026[col].size, null_col_values[col].size)}% null')
# print([percent(houses_sold_2026[col].size, null_col_values[col].size) for col in null_col_values.columns])

Column WaterfrontYN: 173509 null values, 99.95% null
Column BasementYN: 169576 null values, 97.69% null
Column FireplacesTotal: 173591 null values, 100.00% null
Column AboveGradeFinishedArea: 173591 null values, 100.00% null
Column TaxAnnualAmount: 173591 null values, 100.00% null
Column BuilderName: 166582 null values, 95.96% null
Column TaxYear: 173591 null values, 100.00% null
Column BuildingAreaTotal: 159003 null values, 91.60% null
Column ElementarySchoolDistrict: 173591 null values, 100.00% null
Column BelowGradeFinishedArea: 172498 null values, 99.37% null
Column CoveredSpaces: 173591 null values, 100.00% null
Column LotSizeDimensions: 161344 null values, 92.94% null
Column MiddleOrJuniorSchoolDistrict: 173591 null values, 100.00% null


In [57]:
# Cutting columns that are completely null or have too many null values. I consider the acceptable cutoff to be around 95%, generally
# However, I'm considering leaving in business type (the sample size is so small anyways...)
# WaterfrontYN and BasementYN are easily inferrable. Even if they make up only a small sample of the data I think it's still helpful to have 
# Additionally, BuilderName is not that relevant, and 96% of listings lack it

prev_size = len(houses_sold_2026)
print(f"Previous size: {prev_size}")
print(houses_sold_2026.columns)
houses_sold_2026 = houses_sold_2026.drop(columns=['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 
                                                  'TaxYear', 'ElementarySchoolDistrict', 'BelowGradeFinishedArea', 
                                                  'CoveredSpaces', 'LotSizeDimensions','MiddleOrJuniorSchoolDistrict'])
print(f"New size: {len(houses_sold_2026)}, -{percent(prev_size, len(houses_sold_2026))}%")

Previous size: 173591
Index(['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
       'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'FireplacesTotal',
       'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'TaxAnnualAmount',
       'CountyOrParish', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'SubdivisionName',
       'YearBuilt', 'BathroomsTotalInteger', 'City', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict',
       'BelowGradeFinishedArea', 'StateOrProvince', 'CoveredSpaces',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool',
       'Levels', 'LotSizeDimensions', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict'],
      dtype='object')
New size: 173591, -0.0%


In [58]:
# Dropping duplicate values and entries lacking the target value

prev_size = len(houses_sold_2026)
# Drop duplicate values (specifically those with matching unparsed address and close date)
houses_sold_2026 = houses_sold_2026.drop_duplicates(['UnparsedAddress', 'CloseDate'])
print_change('Address and Date Duplicates', prev_size, len(houses_sold_2026))

# Drop entries that lack closeprice or otherwise lack close price
zero_prices = (houses_sold_2026[houses_sold_2026['ClosePrice']<= 0].index)
houses_sold_2026 = houses_sold_2026.drop(zero_prices)
houses_sold_2026 = houses_sold_2026.dropna(subset='ClosePrice')
print_change(f'Zero or null prices dropped', prev_size, len(houses_sold_2026))

Address and Date Duplicates: 169, 0.09735527763536128% decrease
Zero or null prices dropped: 180, 0.10369201168263331% decrease


In [59]:
# Dealing with impossible lot size 
# Value counts of lot size by property type and subtype
# Naturally, most are condos. Mostly because the shared spaces make it difficult to deal with
    # Not sure how to immute this just yet tho
# For now, I'll only delete the properties that are not condominimums 
values_of_interest = ['PropertyType', 'PropertySubType', 'ClosePrice']
# print(houses_sold_2026[houses_sold_2026['LotSizeSquareFeet'] <= 0][values_of_interest].value_counts())
print(houses_sold_2026['PropertySubType'].value_counts())
prev_size = len(houses_sold_2026)
# None of the houses are condominiums so I'll just drop all the ones with NA for lotsize
houses_sold_2026 = houses_sold_2026.dropna(subset='LotSizeSquareFeet')
print_change('Removed rows for impossible lot size', prev_size, len(houses_sold_2026))

# All property subtype is singlefamilyresidence, we can remove it now
houses_sold_2026 = houses_sold_2026.drop(columns='PropertySubType')


PropertySubType
SingleFamilyResidence    173411
Name: count, dtype: int64
Removed rows for impossible lot size: 4546, 2.6215176661226796% decrease


note: bedroom/bathroom counts inconsistent with lot size also need to be excised. But i'll figure that out later
also need to figure out any other inconsistencies not listed in the document :( (sad face)

# Split Dataset

In [60]:
# Splitting data
# Easy enough for me to do without sklearn, actually
print(f'Total houses sold: {len(houses_sold_2026)}')

testing_set = houses_sold_2026[houses_sold_2026['CloseDate'].str.contains('2026-05')]
print(f'Length of Testing Set: {len(testing_set)}')

training_set = houses_sold_2026.drop(testing_set.index)
print(f'Length of Training Set: {len(training_set)}')

print(f'Checking Length: {len(testing_set) + len(training_set)} = {len(houses_sold_2026)}')

# Keep close date for training set so we can use "number of months" as a hyperparameter during training
houses_sold_2026 = houses_sold_2026.drop(columns=['CloseDate'])
testing_set = testing_set.drop(columns=['CloseDate'])

Total houses sold: 168865
Length of Testing Set: 14034
Length of Training Set: 154831
Checking Length: 168865 = 168865


In [61]:
# Removing the worst outliers in the dataset (using metrics determined from training dataset)

prev_size = len(training_set) + len(testing_set)
# Getting upper and lower bounds
upper_bound = training_set['ClosePrice'].quantile(0.99)
lower_bound = training_set['ClosePrice'].quantile(0.01)

# removing
training_set = training_set[(training_set['ClosePrice'] > lower_bound) & (training_set['ClosePrice'] < upper_bound)]
testing_set = testing_set[(testing_set['ClosePrice'] > lower_bound) & (testing_set['ClosePrice'] < upper_bound)]
print_change('Removal of outliers', prev_size, len(training_set) + len(testing_set))

Removal of outliers: 3404, 2.0158114470138866% decrease


# Deal with Missing Values

In [62]:
# First, let's see which columns have null / missing values
print(training_set.isnull().sum())

Flooring                    55074
ViewYN                      12633
WaterfrontYN               151627
BasementYN                 148271
PoolPrivateYN               10558
CloseDate                       0
ClosePrice                      0
Latitude                        9
Longitude                       9
UnparsedAddress               106
PropertyType                    0
LivingArea                    154
AssociationFeeFrequency    118446
CountyOrParish                  0
ElementarySchool           132467
AttachedGarageYN            21463
ParkingTotal                    1
SubdivisionName             99212
YearBuilt                     226
BathroomsTotalInteger          12
City                           39
BuildingAreaTotal          138991
BedroomsTotal                   0
StateOrProvince                 1
MiddleOrJuniorSchool       132277
FireplaceYN                   122
Stories                     14568
HighSchool                 126374
Levels                       9373
MainLevelBedro

In [63]:

#print(houses_sold_2026[houses_sold_2026['UnparsedAddress'].isna()][['City', 'SubdivisionName']])

#print(houses_sold_2026[houses_sold_2026['City'].isna()][['UnparsedAddress', 'SubdivisionName', 'PostalCode', 'Latitude', 'Longitude']])
# print(houses_sold_2026['PropertySubType'].value_counts())
#print(houses_sold_2026[['UnparsedAddress', 'City', 'SubdivisionName']])

# Imputing Values

Columns with missing values (and method to fill):
ListAgentAOR, 34: Flag?
Flooring, 58844: Immute with "Unknown"
YN values: Immute with "False"
SubdivisionName: Immute with "Not Listed"
UnparsedAddress, 140: Immute with City
City, 51: Immute with the nonnumber part of the UnparsedAddress

There is one datapoint that doesn't have any address. I guess I'll just delete it.

Latitude, 15: Flag
Longitude, 15: Flag. Use one from the city/UnparsedAddress
LivingArea, 206: Immute with sqfootage
BuyerOfficeName, 1534: 
CoListOfficeName, 120160:
AssociationFeeFrequency, 123628: Immute with NotApplicable
MLSAreaMajor, 20012:
ElementarySchool, 138341: N/A
ParkingTotal, 2: Immute with 0
SubdivisionName, 101168: Use UnparsedAddress
BuyerOfficeAOR, 10306: Idk just yet
YearBuilt, 281: Idk just yet
BathroomsTotalInteger, 14: Idk just yet
City, 51: Use UnparsedAddress
BuildingAreaTotal, 144558: Maybe immute with square footage or smth
MiddleOrJuniorSchool, 138176: Just put in none or N/A
Stories, 15193: Just put in 1
HighSchool, 132209: Either put in NA or flag
Levels, 9477: Just put in Other
LotSizeArea, 4215: idk about this one just yet (condos most likely)
MainLevelBedrooms, 60182: Make it equal to bedroomsTotal
GarageSpaces, 8459: Put in 0
HighSchoolDistrict, 43620: Either put in NA or flag
PostalCode, 22: Lowk could figure it out via unparsed address. Otherwise flag
AssociationFee, 49674: Either put in 0 or NA

In [64]:
# Imputing Values
from sklearn.impute import SimpleImputer

# Imputing and flagging here are not based on means or medians, so I can do it to both without any leakage
for set in [training_set, testing_set]:
    # Imputing YN columns
    yn_cols = [col for col in set.columns if 'YN' in col]
    set.loc[:, yn_cols] = set.loc[:, yn_cols].fillna(False)

    # Imputing Values with suitable placeholder values or flagging
    set['Flooring'] = set['Flooring'].fillna('Unknown')
    set['AssociationFeeFrequency'] = set['AssociationFeeFrequency'].fillna('NotApplicable')
    set['Levels'] = set['Levels'].fillna('Other')
    zero_cols = ['YearBuilt']
    set.loc[:, zero_cols] = set.loc[:, zero_cols].fillna('Unlisted')

        # Schools gets imputed with Other. We treat it like 'none'
    school_cols = [col for col in set.columns if 'School' in col]
    set.loc[:, school_cols] = set.loc[:, school_cols].fillna('Other') 


    # Values that can be imputed with 0
    zero_cols = ['ParkingTotal', 'AssociationFee', 'GarageSpaces','BathroomsTotalInteger', 'MainLevelBedrooms']
    set.loc[:, zero_cols] = set.loc[:, zero_cols].fillna(0)   # Association fee 0 == No fee

    # Values that can be imputed with 1
    zero_cols = ['Stories']
    set.loc[:, zero_cols] = set.loc[:, zero_cols].fillna(1)  


    # Imputing values from other columns
    set['LivingArea'] = set['LivingArea'].fillna(set['LotSizeSquareFeet'])
    set['BuildingAreaTotal'] = set['BuildingAreaTotal'].fillna(set['LotSizeSquareFeet'])

# Imputing address values


#print(training_set[training_set['UnparsedAddress'].isna()][['SubdivisionName', 'Latitude', 'Longitude', 'PostalCode']])
#print(training_set[training_set['MainLevelBedrooms'].isna()][['BedroomsTotal', 'Levels', 'Stories']])

#print(training_set.isnull().sum())
#print(testing_set.isnull().sum())
#tf_imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=False)

#tf_imp.fit_transform(training_set[col for col in training_set.columns if 'YN' in col], )

C:\Users\donutii\AppData\Local\Temp\ipykernel_15264\574829771.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  set.loc[:, yn_cols] = set.loc[:, yn_cols].fillna(False)
C:\Users\donutii\AppData\Local\Temp\ipykernel_15264\574829771.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2021.0 1956.0 1963.0 ... 1995.0 1890.0 1948.0]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  set.loc[:, zero_cols] = set.loc[:, zero_cols].fillna('Unlisted')
C:\Users\donutii\AppData\Local\Temp\ipykernel_15264\574829771.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects

In [65]:
# I would like to use geopy to increase accuracy but I need to configure conda with my device properly first
#from geopy.geocoders import Nominatim

# AI Generated (with human notes to understand code)
# ----- Address-like text fields -----
for df in [training_set, testing_set]: 
# Creating flag for unparsed address
    df["HasUnparsedAddress"] = df["UnparsedAddress"].notna().astype(int)

# Filling unparsed address and subdivision name with unknown (cannot be reliably inferred)
    df["UnparsedAddress"] = df["UnparsedAddress"].fillna("Unknown")
    df["SubdivisionName"] = df["SubdivisionName"].fillna("Unknown")

# ----- Clean postal codes ----- (converting to int)
    df["PostalCode"] = df["PostalCode"].astype(str).str.strip()
    df["PostalCode"] = df["PostalCode"].replace({"nan": np.nan, "None": np.nan})
    df["PostalCode"] = df["PostalCode"].str.replace(r"[^0-9]", "", regex=True)
    df["PostalCode"] = df["PostalCode"].replace("", np.nan)

# ----- Coordinate imputation from training data only -----
# Imputating latitude and longitude based on postal code (and vice versa)
postal_lat = training_set.groupby("PostalCode")["Latitude"].median()
postal_lon = training_set.groupby("PostalCode")["Longitude"].median()

city_lat = training_set.groupby("City")["Latitude"].median()
city_lon = training_set.groupby("City")["Longitude"].median()

# City postal here takes the postal codes from all cities in the training data and picks the most common existing postal code
# generates city's default postal code (to fill postal codes)
city_postal = (
    training_set.groupby("City")["PostalCode"]
    .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
)
# Gets cities from postal codes (basically reverse of previous)
# Generates city from postal code (to fill city values)
city_from_postal = (
    training_set.dropna(subset=["PostalCode", "City"])
    .groupby("PostalCode")["City"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
)


for df in [training_set, testing_set]:
    df["Latitude"] = df["Latitude"].fillna(df["PostalCode"].map(postal_lat))
    df["Longitude"] = df["Longitude"].fillna(df["PostalCode"].map(postal_lon))

    df["Latitude"] = df["Latitude"].fillna(df["City"].map(city_lat))
    df["Longitude"] = df["Longitude"].fillna(df["City"].map(city_lon))

    df["Latitude"] = df["Latitude"].fillna(training_set["Latitude"].median())
    df["Longitude"] = df["Longitude"].fillna(training_set["Longitude"].median())

    df["PostalCode"] = df["PostalCode"].fillna(df["City"].map(city_postal))
    df["PostalCode"] = df["PostalCode"].fillna("Unknown")
    
    df["City"] = df["City"].fillna(df["PostalCode"].map(city_from_postal))
    df["City"] = df["City"].fillna("Unknown")
    
print(training_set.isnull().sum())
#print(testing_set.isnull().sum())


Flooring                   0
ViewYN                     0
WaterfrontYN               0
BasementYN                 0
PoolPrivateYN              0
CloseDate                  0
ClosePrice                 0
Latitude                   0
Longitude                  0
UnparsedAddress            0
PropertyType               0
LivingArea                 0
AssociationFeeFrequency    0
CountyOrParish             0
ElementarySchool           0
AttachedGarageYN           0
ParkingTotal               0
SubdivisionName            0
YearBuilt                  0
BathroomsTotalInteger      0
City                       0
BuildingAreaTotal          0
BedroomsTotal              0
StateOrProvince            1
MiddleOrJuniorSchool       0
FireplaceYN                0
Stories                    0
HighSchool                 0
Levels                     0
MainLevelBedrooms          0
NewConstructionYN          0
GarageSpaces               0
HighSchoolDistrict         0
PostalCode                 0
AssociationFee

# Feature Engineering

Features to Engineer: 
- Property age
- Living ratio
- 

In [66]:
# Importing district info from csv
district_info = pd.read_csv(f"{root}/raw_data/districtdata/CalDistrictAreas.csv")

In [67]:
new_training_set = training_set.copy()
new_testing_set = testing_set.copy()

# Using dictionary lookup to get vals
    # Generated with AI
district_lookup = (
    district_info[["District Name", "Homeless (%)"]]
    .dropna()
    .drop_duplicates("District Name")
    .assign(**{"District Name": lambda x: x["District Name"].astype(str).str.strip().str.upper()})
    .set_index("District Name")["Homeless (%)"]
)

district_cols = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "HighSchoolDistrict",
    "SubdivisionName",
]

def get_homeless(row):
    for col in district_cols:
        val = row[col]
        if pd.notna(val) and not (val == 'Other' or val =='Unknown'):
            val = str(val).strip().upper()
            if val in district_lookup.index:
                return district_lookup.loc[val]
    return np.nan

for data in [new_training_set, new_testing_set]:
    # Engineering simple features
    data['LivingRatio'] = data['LivingArea'] / data['LotSizeSquareFeet']   # Ratio of Living Area to LotSize
    
    # Engineering Features from District Data
    data["Homeless%"] = data.apply(get_homeless, axis=1)
    data['Homeless%'] = data['Homeless%'].fillna(data['Homeless%'].mean())
    #print(data['Homeless%'].value_counts())

# Encoding Values
Using this website as reference https://medium.com/@prathik.codes/how-to-do-target-encoding-without-data-leakage-the-right-way-280bd24fbc81 

Methods:
===
Flooring: Might be too noisy, I'll drop for now and ask about it later

Binary encoding: PropertyType (Residential = 0, ResidentialLease = 1).


Map: 
Levels: 123 for whole values, +0.5 for "ormore", 0 for none, for multisplit/multiple levels add every level together, 
YearBuilt: keep as numeric, or better, convert to age: current_year - YearBuilt.

Target Encoding:
I'll use median because of outliers
CountyOrParish, ElementarySchool, SubdivisionName, City, StateOrProvince, MiddleOrJuniorSchool, HighSchool, HighSchoolDistrict, PostalCode

In [68]:
# Dropping some values that are completely unneeded
    # In the case of flooring, it's difficult to figure out without one hot encoding 
unneeded_cols = ['UnparsedAddress', 'Flooring']
training_set = training_set.drop(columns=unneeded_cols) # not as helpful as city/subdivision/etc
testing_set = testing_set.drop(columns=unneeded_cols) # not as helpful as city/subdivision/etc

print(houses_sold_2026['Levels'].value_counts())


Levels
One                               95099
Two                               55468
ThreeOrMore                        3517
MultiSplit                         2788
One,Two                             383
Two,MultiSplit                      293
Two,One                             282
ThreeOrMore,MultiSplit              150
One,MultiSplit                       96
Two,ThreeOrMore                      62
MultiSplit,One                       32
One,ThreeOrMore                      19
One,Two,MultiSplit                    9
ThreeOrMore,One                       7
Two,ThreeOrMore,MultiSplit            5
One,Two,ThreeOrMore                   4
Two,MultiSplit,One                    3
One,Two,ThreeOrMore,MultiSplit        1
Name: count, dtype: int64


In [69]:
from sklearn.model_selection import KFold



# -----------------------------
# 1) Simple binary / custom map
# -----------------------------
for df in [training_set, testing_set, new_training_set, new_testing_set]:
    df["PropertyType"] = df["PropertyType"].map(
        {"Residential": 0, "ResidentialLease": 1}
    )

def encode_levels(x):
    x = str(x).split(",")
    
    if(len(x) == 1 and x == 'MultiSplit'): return 3.5
    
    mapping = {
        "None": 0.0,
        "One": 1.0,
        "Two": 2.0,
        "TwoOrMore": 2.5,
        "Three": 3.0,
        "ThreeOrMore": 3.5,
        "MultiSplit": 0.5,
        "BiLevel": 2.5
    }
    
    encoded_level = 0.0
    
    for l in x:
        encoded_level += mapping.get(l, 0.0)
        
    return encoded_level 

for df in [training_set, testing_set, new_training_set, new_testing_set]:
    df["Levels"] = df["Levels"].apply(encode_levels)

# -----------------------------
# 2) YearBuilt -> age
# -----------------------------
for df in [training_set, testing_set, new_training_set, new_testing_set]:
    df["YearBuilt"] = (2026 - pd.to_numeric(df["YearBuilt"], errors="coerce")).fillna(0)
    
# -----------------------------
# 3) Target encode categorical columns with KFold
# -----------------------------
def kfold_target_encode(train, test, cat_cols, target_col="ClosePrice", n_splits=5):
    train_enc = train.copy()
    test_enc = test.copy()

    for col in cat_cols:
        global_mean = train[target_col].mean()
        # No shuffling should be done on the data. (I also didn't use random state)
        # Currently using 5 splits. Since the population is quite large I might make it bigger
        kf = KFold(n_splits=n_splits, shuffle=False)

        oof = np.zeros(len(train), dtype=float)

        for tr_idx, va_idx in kf.split(train):
            tr = train.iloc[tr_idx]
            va = train.iloc[va_idx]

            fold_mean = tr.groupby(col)[target_col].mean()
            oof[va_idx] = va[col].map(fold_mean).fillna(global_mean).to_numpy()

        train_mean = train.groupby(col)[target_col].mean()

        train_enc[col] = oof
        test_enc[col] = test[col].map(train_mean).fillna(global_mean)
    
    return train_enc, test_enc


In [70]:
target_cols = [
    "CountyOrParish",
    "ElementarySchool",
    "SubdivisionName",
    "City",
    "StateOrProvince",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "HighSchoolDistrict",
    "PostalCode",
    "AssociationFeeFrequency"
]

training_set, testing_set = kfold_target_encode(
    training_set,
    testing_set,
    target_cols,
    target_col="ClosePrice",
    n_splits=5
)

In [71]:
# Export datasets

# Training sets:
# testing_set = houses_sold_2026[houses_sold_2026['CloseDate'].str.contains('2026-05')]
training_set.to_csv('CRMLS_202505_202604_training_set.csv', index = False)
testing_set.to_csv('CRMLS_202605_testing_set.csv', index = False)

In [72]:
# Encoding the engineered datasets
new_training_set, new_testing_set = kfold_target_encode(
    new_training_set,
    testing_set,
    target_cols,
    target_col="ClosePrice",
    n_splits=5
)

# More feature engineering after encoding was finished
for data in [new_training_set, new_testing_set]:
    data['PropertyAge'] = 2026 - data['YearBuilt']    # Property age
    data['LivingRatio'] = data['LivingArea'] / data['LotSizeSquareFeet']   # Ratio of Living Area to LotSize

In [73]:
# Export new datasets
new_training_set.to_csv('CRMLS_202505_202604_training_set_fe.csv', index = False)
new_testing_set.to_csv('CRMLS_202605_testing_set.csv_fe', index = False)